# 04. Failure Subtypes

This notebook reconstructs the CoT-failure subtype analysis from the public data payload. The archived centroid and composition tables are shown for reference, and a fresh `k=2` clustering is run on G3 trajectories at layer 13.
        


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'data'
FIGURES = ROOT / 'figures'
FIGURES.mkdir(exist_ok=True)
sys.path.insert(0, str(ROOT))

from src.statistical_analysis import cohens_d, two_way_anova, stratified_logistic_auc

sns.set_theme(style='whitegrid', font_scale=1.1)
pd.set_option('display.max_columns', 120)

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

centroids = pd.read_csv(DATA / 'analysis' / 'analysis_B_centroids.csv')
composition = pd.read_csv(DATA / 'analysis' / 'analysis_B_composition.csv')
df = pd.read_csv(DATA / 'qwen05b' / 'unified_metrics.csv')
centroids, composition
        


In [ ]:
features = ['radius_of_gyration', 'effective_dim', 'msd_exponent', 'cos_to_late_window', 'tortuosity', 'speed', 'recurrence_rate']
g3 = df[(df['group'] == 'G3') & (df['layer'] == 13)].dropna(subset=features).copy()
X = StandardScaler().fit_transform(g3[features])
labels = KMeans(n_clusters=2, n_init=20, random_state=42).fit_predict(X)
g3['cluster'] = labels
proj = PCA(n_components=2, random_state=42).fit_transform(X)
g3['pc1'] = proj[:, 0]
g3['pc2'] = proj[:, 1]

plt.figure(figsize=(7, 5))
sns.scatterplot(data=g3, x='pc1', y='pc2', hue='cluster', palette='Set2', s=60)
plt.title('K-means clustering of CoT failures at layer 13')
plt.tight_layout()
plt.savefig(FIGURES / 'repro_04_failure_subtypes.png', dpi=300)
plt.show()

display(g3.groupby('cluster')[features].mean())
print('Archived centroid summary:')
display(centroids)
print('Archived composition summary:')
display(composition)
        
